In [ ]:
%cd ../..
import os
from PIL import Image
import numpy as np
import torch
from collections import defaultdict
from tqdm import tqdm
from omegaconf import OmegaConf

import matplotlib.pyplot as plt

from dinov2.inference import generate_embeddings, build_model, view_volume

In [ ]:
def load_volume(folder_path):
    img_paths = [x for x in os.listdir(folder_path) if x.endswith((".png", ".jpg", ".JPG"))]
    img_paths.sort(key=lambda x: int(x.split(".")[0]))

    size_groups = defaultdict(list)
    for p in img_paths:
        img = Image.open(os.path.join(folder_path, p))
        size_groups[img.size].append(p)

    def process_group(paths):
        images_stack = []
        for p in paths:
            img = Image.open(os.path.join(folder_path, p))
            img = np.array(img)
            images_stack.append(img)

        img = np.stack(images_stack)
        if len(img.shape) == 4:
            img = img.mean(axis=3)

        vmin, vmax = -1000.0, 200.0
        hu = (img / 255.0) * (vmax - vmin) + vmin
        hu = np.clip(hu, -1000, 1900)

        assert ((hu > -1000) & (hu < -800)).any(), "HU volume must contain values below -800"
        assert ((hu >= -100) & (hu <= 50)).any(), "HU volume must contain values between -100 and 50"
        
        num_eq_200 = np.sum(hu == 200)
        num_in_range = np.sum((hu >= -100) & (hu <= 50))
        assert num_eq_200 < num_in_range, (
            f"Density check failed: {num_eq_200} values == 200, "
            f"but only {num_in_range} values in [-100, 50]"
        )

        return hu

    if len(size_groups) == 1:
        hu = process_group(img_paths)
    else:
        valid_groups = {}
        for size, paths in size_groups.items():
            try:
                hu = process_group(paths)
                valid_groups[size] = hu
            except AssertionError:
                continue

        if not valid_groups:
            raise ValueError("No valid image groups found that meet the HU conditions")

        best_size = max(valid_groups.keys(), key=lambda s: s[0]*s[1])
        hu = valid_groups[best_size]

    return torch.from_numpy(hu).float()

In [ ]:
from glob import glob

ncp_paths = glob("/scratch/VM/radio-foundation/datasets-nodicom/CCCII/NCP/**", recursive=True)
ncp_paths = [p for p in ncp_paths if p.count("/") == 8]
len(ncp_paths)

In [ ]:
idx = 8
sample_path = ncp_paths[idx]
img = load_volume(sample_path)
D, W, H = img.shape
plt.imshow(img[D//2], cmap="gray", vmin=-1000, vmax=1000)
plt.colorbar()
plt.show()
print(img.shape)

In [ ]:
plt.hist(img.flatten(), bins=200)
plt.show()

In [ ]:
config_path = "/home/48078029W/projects/radio-foundation/runs/base10pat/config.yaml"
checkpoint_path = "/home/48078029W/projects/radio-foundation/runs/base10pat/eval/training_99999/teacher_checkpoint.pth"

device = torch.device("cuda")

config = OmegaConf.load(config_path)
model, autocast_ctx = build_model(checkpoint_path, config, img_size=504, device=device)

In [ ]:
data_path = "/scratch/VM/radio-foundation/datasets-nodicom/CCCII"
output_path = "/scratch/VM/radio-foundation/cache/embeddings/CCCCII"

In [ ]:
data_kwargs = dict(
    fmean = -573.8,
    fstd = 461.3,
    channels = 10,
    img_size = 504,
    patch_size = 14,
    device="cuda",
    block_size=64,
    autocast_ctx=autocast_ctx,
)
class_names = ["CP", "NCP", "Normal"]
for c in class_names:
    print(c)
    base_path = os.path.join(data_path, c)
    os.makedirs(os.path.join(output_path, c), exist_ok=True)
    for id_i in tqdm(os.listdir(base_path)):
        id_i_path = os.path.join(base_path, id_i)
        for id_j in os.listdir(id_i_path):
            scan_path = os.path.join(id_i_path, id_j)
            new_id = f"{id_i:04}_{id_j:04}"

            p_output_dir = os.path.join(output_path, c, f"{new_id}.pth")
            if os.path.exists(p_output_dir):
                continue

            try:
                img = load_volume(folder_path=scan_path)

                collated_features = generate_embeddings(
                    img,
                    model=model,
                    **data_kwargs # type: ignore
                )

                output = {"cls": collated_features["cls"]}

                torch.save(output, p_output_dir)
            except Exception:
                pass

